This model's purpose is to ues KNN Classification to group different images into two or more clusters, before being able to take a new image and classify them through nearest-neighbour voting. The datasets used are stord in a Dataset folder with the names "Cat" and "Dog".

The dataset was obtained from here: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset?resource=download

In [84]:
# Initial Imports.
from PIL import Image

import numpy as np
import os

In [86]:
# Dataset Class (will hold matricized versions of every image).
class Dataset:
    def __init__(self, imageDimensions = [100, 100]):
        imageDimensions = imageDimensions
        self.load_dataset()

    def load_dataset(self):
        # Should only be run once, as it takes a while.
        self.dataset = {
            "Cat": [],
            "Dog": []
        }

        for folder in self.dataset.keys():
            targetFolder = os.path.join("Datasets", folder)
            for file in os.listdir(targetFolder):
                imagePath = os.path.join(targetFolder, file)
                self.dataset[folder].append(self.convert_image(imagePath))

    def convert_image(self, imagePath):
        image = Image.open(imagePath)
        image = image.resize((imageDimensions[0], imageDimensions[1]))
        image = image.convert("L") # Greyscale to reduce dimension size and minimize colour problems.
        return np.array(image)

In [87]:
# Generate the dataset. This dataset can then be used in as many models as we want.
dataset = Dataset()

In [119]:
# kNN Classification Model
class kNNClassifier:
    def __init__(self, dataset, differenceExponent = 2, kCount = 10):
        self.dataset = dataset
        self.differenceExponent = differenceExponent
        self.kCount = kCount

    def get_image_distance(self, image1, image2):
        difference = image1 - image2

        match self.differenceExponent:
            case 1:
                difference = np.abs(difference)

            case 2:
                difference = np.square(difference)

            case _:
                difference = np.power(difference, self.differenceExponent)
                if self.differenceExponent % 2 != 0:
                    difference = np.abs(difference)

        return np.sum(difference)

    def classify_image(self, imagePath):
        imageData = self.dataset.convert_image(imagePath)
        scores = []

        for classification in self.dataset.dataset.keys():
            for datasetImage in self.dataset.dataset[classification]:
                score = self.get_image_distance(datasetImage, imageData)

                targetIndex = 0
                hasPlaced = False
                
                while not hasPlaced:
                    if targetIndex >= self.kCount:
                        hasPlaced = True

                    elif targetIndex >= len(scores):
                        scores.append([classification, score])
                    
                    elif scores[targetIndex][1] > score:
                        scores.insert(targetIndex, [classification, score])

                    else:
                        targetIndex += 1

        classifications = {}

        for score in scores[:self.kCount]:
            if not score[0] in classifications:
                classifications[score[0]] = 0

            classifications[score[0]] += 1

        resultClassification = dict(sorted(classifications.items(), key = lambda item: item[1], reverse = True))
        return list(resultClassification.keys())[0]

The test dataset was stored in a folder named "\_TEST\_" within the "Datasets" folder. Within it were two more folders, "Cat" and "Dog", each with 250 images each. Each image is run through the classifier, and at the end, its overall score is given as a percentage of images it got right, alongside how many images it got right within each classification group.

The test dataset was obtained here: https://www.kaggle.com/datasets/abdullahbakr7/cats-vs-dogs-image-classification/data. A random 250 images were selected from it.

In [120]:
# Load test dataset.
testImages = {
    "Cat": [],
    "Dog": []
}

for classification in testImages.keys():
    folderPath = os.path.join("Datasets", "_TEST_", classification)

    for image in os.listdir(folderPath):
        testImages[classification].append(os.path.join(folderPath, image))

In [121]:
# Test the classifier.
classifier = kNNClassifier(dataset)
corrects = {}

for classification in testImages.keys():
    corrects[classification] = 0

    for image in testImages[classification]:
        result = classifier.classify_image(image)

        if result == classification:
            corrects[classification] += 1

successPercentages = {}
total = 0
totalCorrects = 0

for classification in corrects.keys():
    successPercentages[classification] = (corrects[classification] / len(testImages[classification])) * 100
    total += len(testImages[classification])
    totalCorrects += corrects[classification]

print("Success Rate: " + str(100 * totalCorrects / total) + "%")

for classification in corrects.keys():
    print(classification + ": " + str(successPercentages[classification]) + "%")

Success Rate: 100.0%
Cat: 100.0%
Dog: 100.0%
